# FIFA World Cup 2026 — Winner Prediction
## Notebook 02 — Feature Engineering

The goal of this notebook is to transform the raw datasets into a clean, 
model-ready feature matrix for training and simulation.

### Objectives
- Clean and prepare all 7 dataframes
- Engineer match-level features
- Merge datasets into a unified feature matrix
- Prepare the 2026 simulation inputs

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

In [2]:
from pathlib import Path

ROOT = Path().resolve().parent
PROCESSED = ROOT / "data" / "processed"

df_matches      = pd.read_csv(PROCESSED / "df_matches_clean.csv")
df_elo          = pd.read_csv(PROCESSED / "df_elo_clean.csv")
df_train        = pd.read_csv(PROCESSED / "df_train_clean.csv")
df_test         = pd.read_csv(PROCESSED / "df_test_clean.csv")
df_matches_2026 = pd.read_csv(PROCESSED / "df_matches_2026_clean.csv")
df_teams_2026   = pd.read_csv(PROCESSED / "df_teams_2026_clean.csv")
df_stages_2026  = pd.read_csv(PROCESSED / "df_stages_2026_clean.csv")
df_host_cities  = pd.read_csv(PROCESSED / "df_host_cities_clean.csv")

print("All dataframes loaded successfully.")

All dataframes loaded successfully.


In [ ]:
# drop irrelevant columns identified during EDA
cols_to_drop = [
    'Key Id', 'Match Id', 'Stadium Id', 'Home Team Id', 'Away Team Id',
    'Match Name', 'tournament Name', 'Match Time', 'Stadium Name',
    'City Name', 'Country Name', 'Home Team Code', 'Away Team Code',
    'Score', 'Replayed', 'Replay'
]

df_matches = df_matches.drop(columns=cols_to_drop)

In [ ]:
# parse Match Date from string to datetime
df_matches['Match Date'] = pd.to_datetime(df_matches['Match Date'])

In [ ]:
# recode legacy group stage formats to standard 'group stage'
# 'second group stage' was used in 1974 and 1982
# 'final round' was used in 1950
df_matches['Stage Name'] = df_matches['Stage Name'].replace({
    'second group stage': 'group stage',
    'final round': 'group stage'
})

df_matches['Stage Name'].value_counts()

Stage Name
group stage          718
round of 16           97
quarter-finals        70
semi-finals           38
final                 21
third-place match     20
Name: count, dtype: int64

In [ ]:
# encode Result as numeric target variable
# 0 = away team win, 1 = draw, 2 = home team win
df_matches['Result'] = df_matches['Result'].map({
    'away team win': 0,
    'draw': 1,
    'home team win': 2
})

df_matches['Result'].value_counts()

Result
2    545
0    240
1    179
Name: count, dtype: int64

In [ ]:
# filter to pre-tournament snapshot only
df_elo = df_elo[df_elo['snapshot_date'] == '2026-05-27']

# drop irrelevant columns
df_elo = df_elo.drop(columns=['year', 'snapshot_date', 'country_code', 
                               'matches_home', 'matches_away', 'matches_neutral'])

df_elo.shape

(48, 17)

In [ ]:
# drop version column — just a dataset version tag, no predictive value
df_train = df_train.drop(columns=['version'])

# drop squad_total_market_value_eur — 32 nulls (~17%), FIFA ranking already captures squad quality
df_train = df_train.drop(columns=['squad_total_market_value_eur'])

df_train.shape

(192, 22)

In [ ]:
# drop same columns as df_train for consistency
df_test = df_test.drop(columns=['version', 'squad_total_market_value_eur'])

df_test.shape

(48, 22)

In [ ]:
# drop irrelevant columns — venue name and airport code not needed for the model
df_host_cities = df_host_cities.drop(columns=['venue_name', 'airport_code'])

df_host_cities.head()

,id,city_name,country,region_cluster
0,1,Atlanta,USA,East
1,2,Boston,USA,East
2,3,Dallas,USA,Central
3,4,Houston,USA,Central
4,5,Kansas City,USA,Central


In [57]:
df_host_cities['region_cluster'].value_counts()

region_cluster
East       6
Central    6
West       4
Name: count, dtype: int64

In [58]:
for name, df in {
    "df_matches": df_matches,
    "df_elo": df_elo,
    "df_matches_2026": df_matches_2026,
    "df_teams_2026": df_teams_2026,
    "df_host_cities": df_host_cities
}.items():
    print(f"\n{name}: {df.columns.tolist()}")


df_matches: ['Tournament Id', 'Stage Name', 'Group Name', 'Group Stage', 'Knockout Stage', 'Match Date', 'Home Team Name', 'Away Team Name', 'Home Team Score', 'Away Team Score', 'Home Team Score Margin', 'Away Team Score Margin', 'Extra Time', 'Penalty Shootout', 'Score Penalties', 'Home Team Score Penalties', 'Away Team Score Penalties', 'Result', 'Home Team Win', 'Away Team Win', 'Draw']

df_elo: ['country', 'rank', 'rating', 'rank_max', 'rating_max', 'rank_avg', 'rating_avg', 'rank_min', 'rating_min', 'matches_total', 'wins', 'losses', 'draws', 'goals_for', 'goals_against', 'confederation', 'is_host']

df_matches_2026: ['id', 'match_number', 'home_team_id', 'away_team_id', 'city_id', 'stage_id', 'kickoff_at', 'match_label']

df_teams_2026: ['id', 'team_name', 'fifa_code', 'group_letter', 'is_placeholder']

df_host_cities: ['id', 'city_name', 'country', 'region_cluster']


## Pre-Merge Column Audit

Before merging, confirmed columns across all dataframes:

- **df_matches** — 21 columns, team names under `Home Team Name` / `Away Team Name`
- **df_elo** — 17 columns, team names under `country`
- **df_matches_2026** — 8 columns, teams referenced by `home_team_id` / `away_team_id`, city by `city_id`
- **df_teams_2026** — 5 columns, team names under `team_name`
- **df_host_cities** — 4 columns, has a `country` column that will conflict with `df_elo` during merges

### Merge Plan for df_matches_2026
1. Rename `id` → `match_id` to avoid conflicts
2. Merge `df_teams_2026` to get home and away team names
3. Merge `df_host_cities` to get `region_cluster`
4. Merge `df_elo` for team Elo features

In [ ]:
# rename id to match_id to avoid conflicts in future merges
df_matches_2026 = df_matches_2026.rename(columns={'id': 'match_id'})

df_matches_2026.columns.tolist()

['match_id',
 'match_number',
 'home_team_id',
 'away_team_id',
 'city_id',
 'stage_id',
 'kickoff_at',
 'match_label']

In [ ]:
# merge home team names
df_matches_2026 = df_matches_2026.merge(
    df_teams_2026[['id', 'team_name', 'fifa_code', 'group_letter']],
    left_on='home_team_id',
    right_on='id',
    how='left'
).drop(columns='id').rename(columns={
    'team_name': 'home_team',
    'fifa_code': 'home_code',
    'group_letter': 'group'
})

df_matches_2026.columns.tolist()

['match_id',
 'match_number',
 'home_team_id',
 'away_team_id',
 'city_id',
 'stage_id',
 'kickoff_at',
 'match_label',
 'home_team',
 'home_code',
 'group']

In [ ]:
# derge away team names
df_matches_2026 = df_matches_2026.merge(
    df_teams_2026[['id', 'team_name', 'fifa_code']],
    left_on='away_team_id',
    right_on='id',
    how='left'
).drop(columns='id').rename(columns={
    'team_name': 'away_team',
    'fifa_code': 'away_code'
})

df_matches_2026.columns.tolist()

['match_id',
 'match_number',
 'home_team_id',
 'away_team_id',
 'city_id',
 'stage_id',
 'kickoff_at',
 'match_label',
 'home_team',
 'home_code',
 'group',
 'away_team',
 'away_code']

In [ ]:
# merge host city info — rename country to host_country to avoid future conflicts with df_elo
df_matches_2026 = df_matches_2026.merge(
    df_host_cities[['id', 'city_name', 'country', 'region_cluster']],
    left_on='city_id',
    right_on='id',
    how='left'
).drop(columns='id').rename(columns={'country': 'host_country'})

df_matches_2026.columns.tolist()

['match_id',
 'match_number',
 'home_team_id',
 'away_team_id',
 'city_id',
 'stage_id',
 'kickoff_at',
 'match_label',
 'home_team',
 'home_code',
 'group',
 'away_team',
 'away_code',
 'city_name',
 'host_country',
 'region_cluster']

In [ ]:
# drop ID columns — no longer needed after merging
df_matches_2026 = df_matches_2026.drop(columns=['home_team_id', 'away_team_id', 'city_id'])

df_matches_2026.columns.tolist()

['match_id',
 'match_number',
 'stage_id',
 'kickoff_at',
 'match_label',
 'home_team',
 'home_code',
 'group',
 'away_team',
 'away_code',
 'city_name',
 'host_country',
 'region_cluster']

In [ ]:
# check home team mismatches
unmatched_home = df_matches_2026[~df_matches_2026['home_team'].isin(df_elo['country'])]['home_team'].dropna().unique()
print("Home mismatches:", unmatched_home)

# check away team mismatches
unmatched_away = df_matches_2026[~df_matches_2026['away_team'].isin(df_elo['country'])]['away_team'].dropna().unique()
print("Away mismatches:", unmatched_away)

Home mismatches: ['USA' "Côte d'Ivoire" 'IR Iran' 'Türkiye' 'Cabo Verde']
Away mismatches: ['Türkiye' 'Cabo Verde' "Côte d'Ivoire" 'IR Iran' 'USA']


In [ ]:
# fix team name mismatches between df_matches_2026 and df_elo
name_fixes = {
    'USA': 'United States',
    "Côte d'Ivoire": 'Ivory Coast',
    'IR Iran': 'Iran',
    'Türkiye': 'Turkey',
    'Cabo Verde': 'Cape Verde'
}

df_matches_2026['home_team'] = df_matches_2026['home_team'].replace(name_fixes)
df_matches_2026['away_team'] = df_matches_2026['away_team'].replace(name_fixes)

# verify
unmatched_home = df_matches_2026[~df_matches_2026['home_team'].isin(df_elo['country'])]['home_team'].dropna().unique()
unmatched_away = df_matches_2026[~df_matches_2026['away_team'].isin(df_elo['country'])]['away_team'].dropna().unique()
print("Home mismatches:", unmatched_home)
print("Away mismatches:", unmatched_away)

Home mismatches: []
Away mismatches: []


In [ ]:
# merge Elo ratings for home team
df_matches_2026 = df_matches_2026.merge(
    df_elo.add_prefix('home_').rename(columns={'home_country': 'home_team'}),
    on='home_team',
    how='left'
)

df_matches_2026.columns.tolist()

['match_id',
 'match_number',
 'stage_id',
 'kickoff_at',
 'match_label',
 'home_team',
 'home_code',
 'group',
 'away_team',
 'away_code',
 'city_name',
 'host_country',
 'region_cluster',
 'home_rank',
 'home_rating',
 'home_rank_max',
 'home_rating_max',
 'home_rank_avg',
 'home_rating_avg',
 'home_rank_min',
 'home_rating_min',
 'home_matches_total',
 'home_wins',
 'home_losses',
 'home_draws',
 'home_goals_for',
 'home_goals_against',
 'home_confederation',
 'home_is_host']

In [ ]:
# merge Elo ratings for away team
df_matches_2026 = df_matches_2026.merge(
    df_elo.add_prefix('away_').rename(columns={'away_country': 'away_team'}),
    on='away_team',
    how='left'
)

df_matches_2026.columns.tolist()

['match_id',
 'match_number',
 'stage_id',
 'kickoff_at',
 'match_label',
 'home_team',
 'home_code',
 'group',
 'away_team',
 'away_code',
 'city_name',
 'host_country',
 'region_cluster',
 'home_rank',
 'home_rating',
 'home_rank_max',
 'home_rating_max',
 'home_rank_avg',
 'home_rating_avg',
 'home_rank_min',
 'home_rating_min',
 'home_matches_total',
 'home_wins',
 'home_losses',
 'home_draws',
 'home_goals_for',
 'home_goals_against',
 'home_confederation',
 'home_is_host',
 'away_rank',
 'away_rating',
 'away_rank_max',
 'away_rating_max',
 'away_rank_avg',
 'away_rating_avg',
 'away_rank_min',
 'away_rating_min',
 'away_matches_total',
 'away_wins',
 'away_losses',
 'away_draws',
 'away_goals_for',
 'away_goals_against',
 'away_confederation',
 'away_is_host']

In [68]:
df_matches_2026.isnull().sum()

match_id               0
match_number           0
stage_id               0
kickoff_at             0
match_label            0
home_team             32
home_code             32
group                 32
away_team             32
away_code             32
city_name              0
host_country           0
region_cluster         0
home_rank             32
home_rating           32
home_rank_max         32
home_rating_max       32
home_rank_avg         32
home_rating_avg       32
home_rank_min         32
home_rating_min       32
home_matches_total    32
home_wins             32
home_losses           32
home_draws            32
home_goals_for        32
home_goals_against    32
home_confederation    32
home_is_host          32
away_rank             32
away_rating           32
away_rank_max         32
away_rating_max       32
away_rank_avg         32
away_rating_avg       32
away_rank_min         32
away_rating_min       32
away_matches_total    32
away_wins             32
away_losses           32


In [ ]:
# confirm nulls are only in knockout stage placeholder matches
df_matches_2026[df_matches_2026['home_team'].isnull()]['match_label'].value_counts()

match_label
2A vs 2B          1
1C vs 2F          1
1E vs 3ABCDF      1
1F vs 2C          1
2E vs 2I          1
1I vs 3CDFGH      1
1A vs 3CEFHI      1
1L vs 3EHIJK      1
1G vs 3AEHIJ      1
1D vs 3BEFIJ      1
1H vs 2J          1
2K vs 2L          1
1B vs 3EFGIJ      1
2D vs 2G          1
1J vs 2H          1
1K vs 3DEIJL      1
W73 vs W75        1
W74 vs W77        1
W76 vs W78        1
W79 vs W80        1
W83 vs W84        1
W81 vs W82        1
W86 vs W88        1
W85 vs W87        1
W89 vs W90        1
W93 vs W94        1
W91 vs W92        1
W95 vs W100       1
W97 vs W98        1
W99 vs W100       1
RU101 vs RU102    1
W101 vs W102      1
Name: count, dtype: int64

In [3]:
# save merged df_matches_2026
# df_matches_2026.to_csv(PROCESSED / "df_matches_2026_merged.csv", index=False)
# print("Saved successfully.")

## Merging df_matches_2026

### Steps
1. Renamed `id` → `match_id` to avoid conflicts
2. Merged `df_teams_2026` for home and away team names
3. Merged `df_host_cities` for `city_name`, `host_country`, `region_cluster` — renamed `country` → `host_country` to avoid future conflicts with `df_elo`
4. Dropped ID columns (`home_team_id`, `away_team_id`, `city_id`) after merging
5. Fixed 5 team name mismatches between `df_matches_2026` and `df_elo`:
   - `USA` → `United States`
   - `Côte d'Ivoire` → `Ivory Coast`
   - `IR Iran` → `Iran`
   - `Türkiye` → `Turkey`
   - `Cabo Verde` → `Cape Verde`
6. Merged `df_elo` for home and away team features using `add_prefix()` — all columns cleanly prefixed with `home_` and `away_`

### Result
- 46 columns, 104 rows
- 32 nulls across team columns — expected, correspond exactly to knockout stage placeholder matches
- Saved to `data/processed/df_matches_2026_merged.csv`

In [ ]:
# check home team mismatches
unmatched_home = df_matches[~df_matches['Home Team Name'].isin(df_elo['country'])]['Home Team Name'].dropna().unique()
print("Home mismatches:", unmatched_home)

Home mismatches: ['Yugoslavia' 'Romania' 'Chile' 'Czechoslovakia' 'Hungary' 'Italy' 'Cuba'
 'West Germany' 'Northern Ireland' 'Soviet Union' 'Wales' 'North Korea'
 'Peru' 'Bulgaria' 'East Germany' 'Zaire' 'Poland' 'Honduras' 'Denmark'
 'United Arab Emirates' 'Costa Rica' 'Cameroon' 'Republic of Ireland'
 'Nigeria' 'Bolivia' 'Russia' 'Greece' 'Jamaica' 'China' 'Slovenia'
 'Trinidad and Tobago' 'Serbia and Montenegro' 'Angola' 'Czech Republic'
 'Togo' 'Ukraine' 'Serbia' 'Slovakia' 'Iceland']


In [ ]:
# check which mismatched teams exist in df_elo under a different name
mismatched = ['Yugoslavia', 'Romania', 'Chile', 'Czechoslovakia', 'Hungary', 'Italy', 'Cuba',
 'West Germany', 'Northern Ireland', 'Soviet Union', 'Wales', 'North Korea',
 'Peru', 'Bulgaria', 'East Germany', 'Zaire', 'Poland', 'Honduras', 'Denmark',
 'United Arab Emirates', 'Costa Rica', 'Cameroon', 'Republic of Ireland',
 'Nigeria', 'Bolivia', 'Russia', 'Greece', 'Jamaica', 'China', 'Slovenia',
 'Trinidad and Tobago', 'Serbia and Montenegro', 'Angola', 'Czech Republic',
 'Togo', 'Ukraine', 'Serbia', 'Slovakia', 'Iceland']

df_elo[df_elo['country'].isin(mismatched)]['country'].tolist()

[]

### Note on df_elo_full
`df_elo` was filtered to the 2026-05-27 snapshot (48 teams only) earlier in the notebook. For merging Elo ratings onto historical matches, the full unfiltered dataset is reloaded as `df_elo_full` — it contains historical snapshots by year, allowing the merge to be done by both team name and year.

In [74]:
# Reload full Elo dataset
df_elo_full = pd.read_csv("../data/processed/df_elo_clean.csv")
df_elo_full.shape

(4683, 23)

In [75]:
df_elo_full.columns.tolist()

['year',
 'snapshot_date',
 'country',
 'rank',
 'country_code',
 'rating',
 'rank_max',
 'rating_max',
 'rank_avg',
 'rating_avg',
 'rank_min',
 'rating_min',
 'matches_total',
 'matches_home',
 'matches_away',
 'matches_neutral',
 'wins',
 'losses',
 'draws',
 'goals_for',
 'goals_against',
 'confederation',
 'is_host']

In [76]:
print("Elo years range:", df_elo_full['year'].min(), "to", df_elo_full['year'].max())
print("Match dates range:", df_matches['Match Date'].min(), "to", df_matches['Match Date'].max())

Elo years range: 1901 to 2026
Match dates range: 1930-07-13 00:00:00 to 2022-12-18 00:00:00


In [ ]:
# extract year from Match Date
df_matches['year'] = df_matches['Match Date'].dt.year

df_matches['year'].value_counts().sort_index()

year
1930    18
1934    17
1938    18
1950    22
1954    26
1958    35
1962    32
1966    32
1970    32
1974    38
1978    38
1982    52
1986    52
1990    52
1994    52
1998    64
2002    64
2006    64
2010    64
2014    64
2018    64
2022    64
Name: count, dtype: int64

In [78]:
unmatched_home = df_matches[~df_matches['Home Team Name'].isin(df_elo_full['country'])]['Home Team Name'].dropna().unique()
print("Home mismatches:", unmatched_home)

Home mismatches: ['Yugoslavia' 'Romania' 'Chile' 'Czechoslovakia' 'Hungary' 'Italy' 'Cuba'
 'West Germany' 'Northern Ireland' 'Soviet Union' 'Wales' 'North Korea'
 'Peru' 'Bulgaria' 'East Germany' 'Zaire' 'Poland' 'Honduras' 'Denmark'
 'United Arab Emirates' 'Costa Rica' 'Cameroon' 'Republic of Ireland'
 'Nigeria' 'Bolivia' 'Russia' 'Greece' 'Jamaica' 'China' 'Slovenia'
 'Trinidad and Tobago' 'Serbia and Montenegro' 'Angola' 'Czech Republic'
 'Togo' 'Ukraine' 'Serbia' 'Slovakia' 'Iceland']


In [79]:
df_elo_full[df_elo_full['country'].isin(unmatched_home)]['country'].unique()

array([], dtype=object)

In [80]:
df_elo_full['country'].sort_values().unique()

array(['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium',
       'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde',
       'Colombia', 'Croatia', 'Curaçao', 'Czechia', 'DR Congo', 'Ecuador',
       'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran',
       'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco',
       'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay',
       'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal',
       'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland',
       'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan'],
      dtype=object)

In [ ]:
# keep only matches where both teams are in the 2026 qualified nations
qualified_teams = df_elo_full['country'].unique()

df_matches_filtered = df_matches[
    df_matches['Home Team Name'].isin(qualified_teams) &
    df_matches['Away Team Name'].isin(qualified_teams)
]

print(f"Original: {df_matches.shape[0]} matches")
print(f"Filtered: {df_matches_filtered.shape[0]} matches")

Original: 964 matches
Filtered: 418 matches


In [ ]:
# merge Elo ratings for home team by team name and year
df_matches_filtered = df_matches_filtered.merge(
    df_elo_full[['country', 'year', 'rating', 'rank', 'confederation', 'is_host',
                 'rating_max', 'rating_avg', 'rating_min',
                 'wins', 'losses', 'draws', 'goals_for', 'goals_against', 'matches_total']],
    left_on=['Home Team Name', 'year'],
    right_on=['country', 'year'],
    how='left'
).drop(columns='country').rename(columns={
    'rating': 'home_elo', 'rank': 'home_rank', 'confederation': 'home_conf',
    'is_host': 'home_is_host', 'rating_max': 'home_elo_max', 'rating_avg': 'home_elo_avg',
    'rating_min': 'home_elo_min', 'wins': 'home_wins', 'losses': 'home_losses',
    'draws': 'home_draws', 'goals_for': 'home_goals_for', 'goals_against': 'home_goals_against',
    'matches_total': 'home_matches_total'
})

df_matches_filtered.shape

(418, 35)

In [ ]:
# merge Elo ratings for away team by team name and year
df_matches_filtered = df_matches_filtered.merge(
    df_elo_full[['country', 'year', 'rating', 'rank', 'confederation', 'is_host',
                 'rating_max', 'rating_avg', 'rating_min',
                 'wins', 'losses', 'draws', 'goals_for', 'goals_against', 'matches_total']],
    left_on=['Away Team Name', 'year'],
    right_on=['country', 'year'],
    how='left'
).drop(columns='country').rename(columns={
    'rating': 'away_elo', 'rank': 'away_rank', 'confederation': 'away_conf',
    'is_host': 'away_is_host', 'rating_max': 'away_elo_max', 'rating_avg': 'away_elo_avg',
    'rating_min': 'away_elo_min', 'wins': 'away_wins', 'losses': 'away_losses',
    'draws': 'away_draws', 'goals_for': 'away_goals_for', 'goals_against': 'away_goals_against',
    'matches_total': 'away_matches_total'
})

df_matches_filtered.shape

(418, 48)

In [84]:
df_matches_filtered.isnull().sum()

Tournament Id                0
Stage Name                   0
Group Name                   0
Group Stage                  0
Knockout Stage               0
Match Date                   0
Home Team Name               0
Away Team Name               0
Home Team Score              0
Away Team Score              0
Home Team Score Margin       0
Away Team Score Margin       0
Extra Time                   0
Penalty Shootout             0
Score Penalties              0
Home Team Score Penalties    0
Away Team Score Penalties    0
Result                       0
Home Team Win                0
Away Team Win                0
Draw                         0
year                         0
home_elo                     0
home_rank                    0
home_conf                    0
home_is_host                 0
home_elo_max                 0
home_elo_avg                 0
home_elo_min                 0
home_wins                    0
home_losses                  0
home_draws                   0
home_goa

## Merging df_matches with df_elo_full

### Key Finding
`df_elo_full` only contains historical snapshots for the 48 teams qualified for 2026 — not all historical nations. Teams like Italy, Yugoslavia, Czechoslovakia, West Germany etc. are not present.

### Decision
Option A — filter `df_matches` to only keep matches where both teams are among the 48 qualified nations. Reduces training set but preserves Elo as a feature.

### Result
- Original: 964 matches
- Filtered: 418 matches — all between currently qualified 2026 teams
- Merged Elo ratings for both home and away teams by team name and year
- Zero nulls across all 48 columns after merge

In [ ]:
# elo difference — positive means home team is stronger
df_matches_filtered['elo_diff'] = df_matches_filtered['home_elo'] - df_matches_filtered['away_elo']

# rank difference — negative means home team is ranked higher
df_matches_filtered['rank_diff'] = df_matches_filtered['home_rank'] - df_matches_filtered['away_rank']

# goal difference
df_matches_filtered['goal_diff'] = df_matches_filtered['Home Team Score'] - df_matches_filtered['Away Team Score']

# win rate for home and away teams
df_matches_filtered['home_win_rate'] = df_matches_filtered['home_wins'] / df_matches_filtered['home_matches_total']
df_matches_filtered['away_win_rate'] = df_matches_filtered['away_wins'] / df_matches_filtered['away_matches_total']

# goals per match for home and away teams
df_matches_filtered['home_goals_per_match'] = df_matches_filtered['home_goals_for'] / df_matches_filtered['home_matches_total']
df_matches_filtered['away_goals_per_match'] = df_matches_filtered['away_goals_for'] / df_matches_filtered['away_matches_total']

df_matches_filtered.shape

(418, 55)

In [86]:
df_matches_filtered[['elo_diff', 'rank_diff', 'goal_diff', 'home_win_rate', 'away_win_rate', 'home_goals_per_match', 'away_goals_per_match']].isnull().sum()

elo_diff                0
rank_diff               0
goal_diff               0
home_win_rate           0
away_win_rate           0
home_goals_per_match    0
away_goals_per_match    0
dtype: int64

In [ ]:
# Save final training feature matrix
# df_matches_filtered.to_csv(PROCESSED / "df_matches_features.csv", index=False)
# print("Saved successfully.")

Saved successfully.


In [ ]:
# elo difference — positive means home team is stronger
df_matches_2026['elo_diff'] = df_matches_2026['home_rating'] - df_matches_2026['away_rating']

# Rank difference — negative means home team is ranked higher
df_matches_2026['rank_diff'] = df_matches_2026['home_rank'] - df_matches_2026['away_rank']

# Win rate for home and away teams
df_matches_2026['home_win_rate'] = df_matches_2026['home_wins'] / df_matches_2026['home_matches_total']
df_matches_2026['away_win_rate'] = df_matches_2026['away_wins'] / df_matches_2026['away_matches_total']

# Goals per match for home and away teams
df_matches_2026['home_goals_per_match'] = df_matches_2026['home_goals_for'] / df_matches_2026['home_matches_total']
df_matches_2026['away_goals_per_match'] = df_matches_2026['away_goals_for'] / df_matches_2026['away_matches_total']

df_matches_2026.shape

(104, 51)

In [89]:
df_matches_2026[['elo_diff', 'rank_diff', 'home_win_rate', 'away_win_rate', 'home_goals_per_match', 'away_goals_per_match']].isnull().sum()

elo_diff                32
rank_diff               32
home_win_rate           32
away_win_rate           32
home_goals_per_match    32
away_goals_per_match    32
dtype: int64

In [ ]:
# Save final 2026 simulation input
# df_matches_2026.to_csv(PROCESSED / "df_matches_2026_features.csv", index=False)
# print("Saved successfully.")

Saved successfully.


In [ ]:
# import os
# files = os.listdir(PROCESSED)
# for f in sorted(files):
#     print(f)

df_elo_clean.csv
df_host_cities_clean.csv
df_matches_2026_clean.csv
df_matches_2026_features.csv
df_matches_2026_merged.csv
df_matches_clean.csv
df_matches_features.csv
df_stages_2026_clean.csv
df_teams_2026_clean.csv
df_test_clean.csv
df_train_clean.csv


In [ ]:
# os.remove(PROCESSED / "df_matches_2026_merged.csv")
# print("Removed intermediate file.")

Removed intermediate file.


## Feature Engineering

### df_matches_filtered (Training Data)
Engineered the following features:
- `elo_diff` — home Elo minus away Elo, positive means home team is stronger
- `rank_diff` — home rank minus away rank, negative means home team is ranked higher
- `goal_diff` — home goals minus away goals
- `home_win_rate` / `away_win_rate` — wins divided by total matches
- `home_goals_per_match` / `away_goals_per_match` — goals for divided by total matches
- Zero nulls across all engineered features
- Saved to `data/processed/df_matches_features.csv`

### df_matches_2026 (Simulation Input)
Same features engineered except `goal_diff` — no matches played yet
- 32 nulls on all engineered features — expected, correspond to knockout placeholders
- Saved to `data/processed/df_matches_2026_features.csv`

### Final processed files
- `df_elo_clean.csv` — full Elo dataset
- `df_host_cities_clean.csv` — host cities with region clusters
- `df_matches_2026_clean.csv` — raw 2026 fixtures
- `df_matches_2026_features.csv` — 2026 fixtures with all features, ready for simulation
- `df_matches_clean.csv` — cleaned historical matches
- `df_matches_features.csv` — historical matches with Elo features, ready for model training
- `df_stages_2026_clean.csv` — tournament stage progression
- `df_teams_2026_clean.csv` — all 48 qualified teams
- `df_test_clean.csv` — 2026 team features test set
- `df_train_clean.csv` — historical team features training set